In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)

In [3]:
citation

{'doi': 'https://doi.org/10.1038/s41467-020-14936-3',
 'citation': 'Wagner, M., Yoshihara, M., Douagi, I. et al. Single-cell analysis of human ovarian cortex identifies distinct cell populations but no oogonial stem cells. Nat Commun 11, 1147 (2020).',
 'table': 'https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-020-14936-3/MediaObjects/41467_2020_14936_MOESM3_ESM.xlsx'}

## Define the metadata

In [4]:
organism = "homo_sapiens"
cell_source = "ovary"
cell_state = None
table_name = "clean_degs.xlsx" # local name

## Read in file

In [5]:
df = pd.read_excel(table_name, skiprows=2, sheet_name=1)

df

,gene,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster
0,FIGLA,0.000000e+00,2.388708,0.679,0.007,0.000000e+00,oocytes
1,KPNA7,0.000000e+00,2.335580,0.821,0.004,0.000000e+00,oocytes
2,NLRP5,0.000000e+00,2.243391,0.714,0.005,0.000000e+00,oocytes
3,C15orf60,0.000000e+00,2.106073,0.643,0.004,0.000000e+00,oocytes
4,ZAR1,0.000000e+00,2.085036,0.714,0.004,0.000000e+00,oocytes
...,...,...,...,...,...,...,...
10205,GAS6,1.271911e-16,-0.274964,0.488,0.483,3.186265e-12,stroma
10206,C8orf4,4.846264e-14,-0.270064,0.405,0.294,1.214038e-09,stroma
10207,NR4A2,3.382492e-12,0.259664,0.191,0.143,8.473481e-08,stroma
10208,NR4A1,1.263942e-08,0.308894,0.657,0.609,3.166302e-04,stroma


## Unfiltered

In [6]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"cluster": "cell_type_label", "gene": "feature_name"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None

unfiltered_df.drop(['p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj'], axis=1, inplace=True) 
result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

## Save unfiltered

In [7]:
with open("evidence_unfiltered.json", 'w') as f:
    json.dump(save, f, indent=4)

## Perform the filtering

In [12]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "cluster"
col_rank = "pct.1"

In [13]:
min_mean = 15
max_pval = 0.05
min_lfc = 1.5
max_gene_shares = 10
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")


# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [14]:
markers[col_ct].value_counts()

cluster
perivascular cells    20
monocytes             20
oocytes               20
endothelial cells     20
t cells               20
granulosa cells       10
stroma                 1
Name: count, dtype: int64

In [15]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
granulosa cells       0.78230
perivascular cells    0.79420
oocytes               0.81235
endothelial cells     0.83625
monocytes             0.83640
t cells               0.91450
stroma                0.96300
Name: pct.1, dtype: float64

## Find Ensembl ID

In [16]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [17]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=3):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [18]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json



In [19]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)
result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

In [20]:
save

[{'cell_type_label': 'granulosa cells',
  'gene': 'HSPA6',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000173110'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'PHLDA1',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000139289'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'NDUFA4L2',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000185633'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'RRAD',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000166592'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'MT1M',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000205364'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'CRIP1',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': No

## Save evidence

In [21]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 